# AlphaEarth — download cache for BloomBench

Populate the local AlphaEarth HDF5 store for every dataset in [`bloombench.config`](../src/pysephone/benchmarks/bloombench/config.py). Resumable — re-running skips locations/years already on disk.

Flow:

1. **Setup** — import shared config so the same dataset list as BloomBench is used.
2. **Parameters** — pick the year range, Earth Engine project, batch size, and optional dataset subset.
3. **Plan** — load all selected datasets, merge their unique locations into one set, and report how many `(location, year)` pairs need to be fetched from Earth Engine. The merge step is the win: a station shared by `PEP725_Apple` and `PEP725_Pear` becomes **one** EE sample, not two.
4. **Download** — single batched `fetch_alphaearth_embeddings_batched` call. Uses `reduceRegions` server-side so each batch of up to 500 points is one round trip.
5. **Verify** — list the HDF5 store path/size and re-count cached entries.

**Prerequisites:**
- One-time Earth Engine auth: `python -c "import ee; ee.Authenticate()"`.
- A GCP project with Earth Engine enabled (default `phenologyembeddings`, override in the params cell).
- `earthengine-api` and `h5py` installed (already in `pyproject.toml`).

## 1. Setup — import shared config

In [ ]:
from __future__ import annotations

import math
import time
from pathlib import Path

import pandas as pd

from pysephone.benchmarks.bloombench import config as bc
from pysephone.data.alphaearth.obtain_embeddings import (
    EMBED_DIM,
    AlphaEarthEmbeddingStore,
    _stable_location_id,
    fetch_alphaearth_embeddings_batched,
)
from pysephone.dataset.dataset import Dataset
from pysephone.dataset.util.calendar import Calendar

_store_preview = AlphaEarthEmbeddingStore()
print(f'Datasets requested: {len(bc.DATASETS_REQUESTED)}')
print(f'Embedding dim:      {EMBED_DIM}')
print(f'Store path:         {_store_preview.h5_path}')

## 2. Parameters

Edit these to control what gets fetched. For a first run, set `DATASETS_SUBSET` to a small dataset (e.g. `['GMU_Cherry_Switzerland']`) to confirm Earth Engine credentials and the GCP project work end-to-end before kicking off the full BloomBench download.

In [2]:
# Earth Engine project (GCP project with EE enabled).
EE_PROJECT = 'phenologyembeddings'

# AlphaEarth annual coverage. Fetching the full range gives downstream code
# free choice of year_offset (e.g. -1 for seasons spanning two calendar years).
YEARS = list(range(2017, 2025))

# location_id rounding (~0.11m at the equator). Must match the precision used
# by AlphaEarthFeatures when reading the store back.
PRECISION = 6

# Native AlphaEarth pixel scale in metres.
SCALE = 10

# Points per reduceRegions call. 200–1000 is the safe range.
BATCH_SIZE = 500

# Subset filter — leave empty for all bench_config datasets, or list a few
# names (e.g. ['GMU_Cherry_Switzerland']) for a smoke test before the full run.
DATASETS_SUBSET: list[str] = []

# Calendar settings (only affect Dataset.load; embeddings are annual and
# independent of season window, but we keep these identical to the AgERA5
# notebook for consistency).
SEASON_START  = bc.SEASON_START
SEASON_LENGTH = bc.SEASON_LENGTH

print('EE project: ', EE_PROJECT)
print('Years:      ', f'{YEARS[0]}..{YEARS[-1]}  ({len(YEARS)} years)')
print('Subset:     ', DATASETS_SUBSET or '(all)')
print('Batch size: ', BATCH_SIZE)
print('Precision:  ', PRECISION)
print('Scale:      ', f'{SCALE} m')

EE project:  phenologyembeddings
Years:       2017..2024  (8 years)
Subset:      (all)
Batch size:  500
Precision:   6
Scale:       10 m


## 3. Plan — merge locations, count EE requests

Loads every selected dataset, builds the union of required `(location, year)` pairs, and reports how many still need an Earth Engine round trip. **No EE calls.** Re-run freely; it just reads the local HDF5 store to count what's missing.

Bundling at this combined level is what saves requests across overlapping species (the 7 PEP725 species sharing a station set become one set of EE batches, not seven).

In [3]:
datasets_to_run = [
    (name, target) for name, target in bc.DATASETS_REQUESTED
    if not DATASETS_SUBSET or name in DATASETS_SUBSET
]
print(f'Selected datasets: {len(datasets_to_run)}')

# location_id -> (lat, lon); deduplicated across all datasets
latlons: dict[str, tuple[float, float]] = {}
per_dataset_rows = []
for ds_name, _target in datasets_to_run:
    cal = Calendar(default_start=SEASON_START, default_length=SEASON_LENGTH)
    ds = Dataset.load(ds_name, calendar=cal, feature_providers=[])

    seen_in_ds: set[str] = set()
    before = len(latlons)
    for ix in ds.observations.iter_index():
        src, loc_id = ix[0], ix[1]
        coords = ds.observations.get_location_coordinates((src, loc_id))
        lid = _stable_location_id(coords['lat'], coords['lon'], precision=PRECISION)
        if lid in seen_in_ds:
            continue
        seen_in_ds.add(lid)
        latlons.setdefault(lid, (float(coords['lat']), float(coords['lon'])))
    added = len(latlons) - before
    per_dataset_rows.append(dict(
        dataset=ds_name,
        unique_locs=len(seen_in_ds),
        unique_added=added,
    ))

per_dataset_df = pd.DataFrame(per_dataset_rows)
print()
print(per_dataset_df.to_string(index=False))

# Cache scan — no EE calls
store = AlphaEarthEmbeddingStore()
missing_per_year: dict[int, int] = {y: 0 for y in YEARS}
cached = 0
with store._open('r') as f:
    for lid in latlons:
        for y in YEARS:
            if f'v1/locations/{lid}/embeddings/{y}' in f:
                cached += 1
            else:
                missing_per_year[y] += 1

needed = len(latlons) * len(YEARS)
missing = needed - cached
ee_batches = sum(math.ceil(m / BATCH_SIZE) for m in missing_per_year.values() if m > 0)

print()
print(f'Unique locations across all datasets:    {len(latlons):>8,}')
print(f'Total (location, year) pairs needed:     {needed:>8,}')
print(f'Already cached on disk:                  {cached:>8,}')
print(f'Missing — needs EE:                      {missing:>8,}')
print(f'Approx. EE batches required:             {ee_batches:>8,}')

Selected datasets: 18


Checking for missing PEP725 data: 100%|██████████| 174/174 [00:00<00:00, 2607.32it/s]



               dataset  unique_locs  unique_added
          PEP725_Apple          834           834
           PEP725_Pear          590           147
          PEP725_Peach          391           189
         PEP725_Almond           20            16
        PEP725_Apricot           69            41
           PEP725_Plum          620           172
   PEP725_Cherry_NoGMU         1056           234
    GMU_Cherry_Japan_Y           67            67
    GMU_Cherry_Japan_S            8             8
GMU_Cherry_Switzerland           67            67
GMU_Cherry_South_Korea           52            52
         USA_NPN_Apple          106           106
          USA_NPN_Pear           66            55
         USA_NPN_Peach           41            29
        USA_NPN_Almond            2             2
       USA_NPN_Apricot            8             4
          USA_NPN_Plum           47            36
        USA_NPN_Cherry          208           192

Unique locations across all datasets:       2,25

## 4. Download — one batched call

Single `fetch_alphaearth_embeddings_batched` call covering every dataset's missing locations across all `YEARS`. Cached `(location, year)` pairs are skipped automatically; restarting the kernel and re-running picks up where it left off (the plan cell shows the new estimate).

tqdm progress bars from the fetcher print inline. Earth Engine `reduceRegions` is server-side, so each batch is one `getInfo()` round trip plus EE's own processing time.

In [4]:
latlons_list = list(latlons.values())  # [(lat, lon), ...]
t0 = time.time()
fetch_alphaearth_embeddings_batched(
    latlons=latlons_list,
    years=YEARS,
    store=store,
    precision=PRECISION,
    scale=SCALE,
    batch_size=BATCH_SIZE,
    ee_project=EE_PROJECT,
)
elapsed = time.time() - t0
print(f'\nTotal wall time: {elapsed/60:.1f} min  ({elapsed:.0f} s)')


Total wall time: 4.6 min  (276 s)


## 5. Verify cache state

Check the HDF5 store on disk. After a successful run, expect `cached_after` to equal `len(latlons) * len(YEARS)` minus any locations that have no AlphaEarth coverage (e.g. open water — the fetcher silently skips those).

In [5]:
store_path = Path(store.h5_path)
print(f'Store path: {store_path}')
if store_path.exists():
    print(f'Store size: {store_path.stat().st_size / 1024 / 1024:.1f} MB')
else:
    print('  (store file does not exist yet — run the download cell first)')

with store._open('r') as f:
    cached_after = sum(
        1 for lid in latlons for y in YEARS
        if f'v1/locations/{lid}/embeddings/{y}' in f
    )
total = len(latlons) * len(YEARS)
pct = (100 * cached_after / total) if total else 0.0
print(f'Cached now: {cached_after:,} / {total:,} (location, year) pairs ({pct:.1f}%)')

Store path: C:\Users\bree026\Repositories\pysephone\data\products\alphaearth\alphaearth_embeddings.h5
Store size: 130.9 MB
Cached now: 18,008 / 18,008 (location, year) pairs (100.0%)
